# Gold Layer — Dimension: Product
## SalesFlow Data Lakehouse | Phase 5: Analytical Layer

Joins `salesflow_dev.silver.products` and `salesflow_dev.silver.categories`
(VALID records only), builds the product dimension with a surrogate key,
and writes to `salesflow_dev.gold.dim_product`.

**Source tables:**
| Table | Role |
|---|---|
| `salesflow_dev.silver.products` | Main source |
| `salesflow_dev.silver.categories` | Lookup for `category_name` |

**Design:**
| Column | Type | Description |
|---|---|---|
| `product_key` | PK | MD5 surrogate key derived from `ProductID` |
| `product_id` | NK | Natural key from source system |
| `product_name` | string | Cleaned product name |
| `category_name` | string | Category from JOIN with categories |
| `unit_price` | double | Unit price |
| `price_category` | string | Low / Medium / High classification |
| `is_available` | boolean | TRUE if units in stock > 0 |
| `effective_date` | date | Date this record was loaded into Gold |

In [0]:
%run ../04_Utils/common_functions

## 1. Read from Silver (VALID records only)

In [0]:
from pyspark.sql.functions import current_date, col

# Read only VALID products from Silver
df_products = spark.table("salesflow_dev.silver.products") \
                   .filter(col("data_quality_status") == "VALID")

# Read only VALID categories from Silver
df_categories = spark.table("salesflow_dev.silver.categories") \
                     .filter(col("data_quality_status") == "VALID")

print(f"Valid products: {df_products.count()}")
print(f"Valid categories: {df_categories.count()}")

## 2. Join Products with Categories
Left join to retain products even if their category is missing.  
`category_name` will be `null` for unmatched products — these are still valid dimension members.

In [0]:
# Left join: keep all valid products, enrich with category name where available
df = df_products.join(
    df_categories.select(
        col("CategoryID"),
        col("CategoryName").alias("category_name")
    ),
    on="CategoryID",
    how="left"
)

print(f"Records after join: {df.count()}")

# Sanity check — how many products have no matching category
unmatched = df.filter(col("category_name").isNull()).count()
print(f"Products with no category match: {unmatched}")

## 3. Select and Rename Columns
Rename to snake_case convention used across the Gold layer.

In [0]:
# Select only the columns needed for the dimension and rename to snake_case
df = df.select(
    col("ProductID").alias("product_id"),
    col("ProductName").alias("product_name"),
    col("category_name"),
    col("UnitPrice").alias("unit_price"),
    col("price_category"),
    col("is_available")
)

## 4. Add Surrogate Key
Generates `product_key` as an MD5 hash of `product_id`.  
The hash is deterministic — same `product_id` always produces the same key.

In [0]:
# Add surrogate key based on natural key product_id
df = add_surrogate_key(df, "product", ["product_id"])

## 5. Add Effective Date

In [0]:
# effective_date marks when this record entered the Gold layer
df = df.withColumn("effective_date", current_date())

## 6. Final Column Order
Enforce the defined schema order: PK first, NK second, attributes, metadata last.

In [0]:
# Enforce final column order as per dimension design
df = df.select(
    "product_key",
    "product_id",
    "product_name",
    "category_name",
    "unit_price",
    "price_category",
    "is_available",
    "effective_date"
)

print(f"Total records in dimension: {df.count()}")
display(df.limit(5))

## 7. Save as Delta Table

In [0]:
# Write to Gold layer as Delta table — overwrite for first load
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("salesflow_dev.gold.dim_product")

print("Table saved: salesflow_dev.gold.dim_product")

## 8. Validation

In [0]:
dim_product = spark.table("salesflow_dev.gold.dim_product")

# Record count
print(f"Total records: {dim_product.count()}")

# Surrogate key uniqueness check — must be 0 duplicates
duplicate_keys = dim_product.groupBy("product_key").count().filter(col("count") > 1)
print(f"\nDuplicate surrogate keys (expected 0): {duplicate_keys.count()}")

# Distribution by category — useful sanity check
print("\nRecords by category:")
display(dim_product.groupBy("category_name").count().orderBy(col("count").desc()))

# Distribution by price category
print("\nRecords by price category:")
display(dim_product.groupBy("price_category").count().orderBy("price_category"))

# Availability distribution
print("\nAvailability distribution:")
display(dim_product.groupBy("is_available").count())

# Schema
print("\nSchema:")
dim_product.printSchema()

# Sample
print("\nFirst 5 rows:")
display(dim_product.limit(5))